# Module 3 Lab 3 — MNIST Evaluation & Visualization

**Module 3: Neural Networks & PyTorch | Week 1 | Lab (Session 3.6)**

---

Getting 97% accuracy sounds great, but that means ~300 test images are wrong. Which ones? Why? How confident is the model when it's right vs when it's wrong?

This lab focuses on **understanding** your model's behavior, not just its overall accuracy.

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
# Load data and rebuild the model from Lab 2
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = datasets.MNIST('data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('data', train=False, transform=transform)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Rebuild and train the model (or load from Lab 2 if saved)
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
).to(device)

# Quick train
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

model.train()
for epoch in range(5):
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(images), labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} done")

print("Training complete.")

---

## Part 1: Collect All Predictions

### Task 1: Get predictions and confidences for the entire test set

In [ ]:
# Collect all predictions, confidences, and true labels
model.eval()
all_images = []
all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        output = model(images.to(device))
        probs = torch.softmax(output, dim=1).cpu()
        preds = output.argmax(dim=1).cpu()

        all_images.append(images)
        all_labels.append(labels)
        all_preds.append(preds)
        all_probs.append(probs)

all_images = torch.cat(all_images)
all_labels = torch.cat(all_labels)
all_preds = torch.cat(all_preds)
all_probs = torch.cat(all_probs)

correct_mask = all_preds == all_labels
accuracy = correct_mask.float().mean().item()

print(f"Overall accuracy: {accuracy:.2%}")
print(f"Correct: {correct_mask.sum().item()} / {len(all_labels)}")
print(f"Wrong: {(~correct_mask).sum().item()}")

---

## Part 2: Prediction Grid with Confidence

### Task 2: Show correct predictions with confidence scores

Display a 4×6 grid of correctly classified images with green borders and the model's confidence.

In [ ]:
# Task 2: Correct predictions grid
correct_indices = torch.where(correct_mask)[0][:24]

fig, axes = plt.subplots(4, 6, figsize=(14, 10))
for i, ax in enumerate(axes.flat):
    idx = correct_indices[i].item()
    img = all_images[idx].squeeze()
    pred = all_preds[idx].item()
    conf = all_probs[idx, pred].item()

    ax.imshow(img, cmap='gray')
    ax.set_title(f"{pred} ({conf:.0%})", fontsize=11, color="#2d8f2d", fontweight="bold")

    # Green border
    for spine in ax.spines.values():
        spine.set_edgecolor('#2d8f2d')
        spine.set_linewidth(3)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("Correct Predictions (with confidence)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### Task 3: Show ALL incorrect predictions

Display incorrect predictions with red borders, showing what the model predicted vs the true label.

In [ ]:
# Task 3: Wrong predictions grid
wrong_indices = torch.where(~correct_mask)[0]
n_wrong = min(len(wrong_indices), 24)

n_cols = 6
n_rows = (n_wrong + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 2.5 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)

for i, ax in enumerate(axes.flat):
    if i >= n_wrong:
        ax.axis('off')
        continue

    idx = wrong_indices[i].item()
    img = all_images[idx].squeeze()
    pred = all_preds[idx].item()
    true = all_labels[idx].item()
    conf = all_probs[idx, pred].item()

    ax.imshow(img, cmap='gray')
    ax.set_title(f"Pred: {pred} ({conf:.0%})\nTrue: {true}",
                 fontsize=10, color="#cc3333", fontweight="bold")

    # Red border
    for spine in ax.spines.values():
        spine.set_edgecolor('#cc3333')
        spine.set_linewidth(3)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle(f"Incorrect Predictions ({n_wrong} total)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

Look at the mistakes. Many of them are digits that are genuinely hard to read — even humans might get them wrong.

---

## Part 3: Per-Digit Accuracy

### Task 4: Which digits are hardest?

Calculate accuracy for each digit (0-9) separately.

In [ ]:
# Task 4: Per-digit accuracy
print(f"{'Digit':>5} {'Correct':>8} {'Total':>6} {'Accuracy':>9}")
print("-" * 35)

digit_accs = []
for digit in range(10):
    mask = all_labels == digit
    n_total = mask.sum().item()
    n_correct = (correct_mask & mask).sum().item()
    acc = n_correct / n_total
    digit_accs.append(acc)
    print(f"{digit:>5} {n_correct:>8} {n_total:>6} {acc:>9.2%}")

In [ ]:
# Bar chart of per-digit accuracy
colors = ['#5CB85C' if a > 0.97 else '#F0AD4E' if a > 0.95 else '#D9534F'
          for a in digit_accs]

plt.figure(figsize=(10, 5))
bars = plt.bar(range(10), [a * 100 for a in digit_accs], color=colors,
               edgecolor='white', linewidth=2)

for bar, acc in zip(bars, digit_accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f"{acc:.1%}", ha='center', fontsize=10, fontweight='bold')

plt.xlabel("Digit", fontsize=12)
plt.ylabel("Accuracy (%)", fontsize=12)
plt.title("Per-Digit Accuracy", fontsize=14, fontweight="bold")
plt.xticks(range(10))
plt.ylim(90, 100.5)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---

## Part 4: Confusion Matrix

### Task 5: Build a confusion matrix

You learned about confusion matrices in Module 2. Now build one for all 10 digits.

In [ ]:
# Task 5: Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(all_labels.numpy(), all_preds.numpy())

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(cm, display_labels=range(10))
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title("MNIST Confusion Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

Look at the off-diagonal entries — those are the mistakes. Which digit pairs get confused most often?

---

## Part 5: Confidence Analysis

### Task 6: Is the model more confident when it's correct?

Compare the model's confidence (max probability) on correct vs incorrect predictions.

In [ ]:
# Task 6: Confidence distribution
max_probs = all_probs.max(dim=1).values.numpy()
correct_conf = max_probs[correct_mask.numpy()]
wrong_conf = max_probs[~correct_mask.numpy()]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(correct_conf, bins=50, alpha=0.7, label=f"Correct (n={len(correct_conf)})",
        color="#5CB85C", edgecolor="white")
ax.hist(wrong_conf, bins=50, alpha=0.7, label=f"Wrong (n={len(wrong_conf)})",
        color="#D9534F", edgecolor="white")
ax.set_xlabel("Model Confidence (max probability)", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
ax.set_title("Confidence Distribution: Correct vs Wrong",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean confidence (correct): {correct_conf.mean():.3f}")
print(f"Mean confidence (wrong):   {wrong_conf.mean():.3f}")

A well-calibrated model should be less confident when it's wrong. If the wrong predictions also have high confidence, the model is overconfident — it's making mistakes but doesn't "know" it.

---

## Challenge: Most uncertain predictions

Find the 10 test images where the model is **most uncertain** (lowest max probability). Display them with all 10 class probabilities as a bar chart next to each image.

In [ ]:
# Challenge: Your code here


---

## Wrap-Up

Evaluation isn't just "what's the accuracy." Understanding *how* your model fails is just as important:

- **Prediction grids** with color-coded borders show correct vs wrong at a glance
- **Per-digit accuracy** reveals which classes are harder
- **Confusion matrices** show which classes get confused with each other
- **Confidence analysis** tells you whether the model "knows" when it's unsure

These evaluation techniques apply to any classification model, not just MNIST.

---

**Next session:** Hyperparameters — how to tune your network's settings for better performance.